# Section 06 — LLM Evaluators & Red Teaming

This section is about testing your AI — not just building it. Once you've added guardrails, agents, and RAG, how do you know it's actually working correctly? That's what evaluators are for.

## What is an LLM Evaluator?

An LLM Evaluator is a tool that **automatically checks if your AI's answer is correct** — without you having to read every single response manually.

Think of it like this:
- You deploy an AI chatbot
- It answers 1000 questions per day
- You can't read all 1000 answers
- So you use an **evaluator** that reads them for you and gives each answer a score

**How does it work?**

Another LLM (the "judge") reads the answer and scores it. This is called an **LLM-as-a-judge** pattern.

```
User question  →  Your AI  →  Answer
                                ↓
                         LLM Judge scores it
                                ↓
                         Score: 0.8 / 1.0  ✅
```

## Does this fit in AI Security?

Yes — and here's why.

All the things we built in previous sections (guardrails, agents, RAG) can fail silently. The AI won't crash — it will just give a wrong or dangerous answer. Without an evaluator, you won't even know.

**3 security problems evaluators catch:**

**1. Hallucination**
- AI makes up information that isn't in your documents
- Example: Customer asks about your refund policy. AI says "30 days" but your policy says "7 days". No error, just wrong.
- Evaluator catches this by checking: *does the answer match the source documents?*

**2. Prompt injection that bypassed your guardrail**
- An attack got through and changed the AI's behavior
- Evaluator catches this by checking: *does the answer match what it should say?*

**3. Data leakage in RAG**
- Your AI is searching documents and accidentally leaking info the user shouldn't see
- Evaluator catches this by checking: *is sensitive data appearing in outputs?*

## The Faithfulness Metric

**Faithfulness** = does the AI's answer stick to the retrieved context, or did it make something up?

This is the most important metric for RAG systems.

**Example:**

Context retrieved from your docs:
> *"Employees can claim up to ₹5000 for travel per month."*

Question: *"How much travel allowance do I get?"*

| AI Answer | Faithful? | Score |
|-----------|-----------|-------|
| "You can claim up to ₹5000 for travel per month." | ✅ Yes | 1.0 |
| "You can claim up to ₹10,000 for travel per month." | ❌ No — hallucination | 0.0 |
| "Travel allowance is ₹5000. You also get ₹2000 for food." | ❌ Partially — food part made up | 0.5 |

**How the judge scores it:**
1. Break the answer into small claims ("travel allowance is ₹5000", "food allowance is ₹2000")
2. Check each claim against the context
3. Score = claims supported / total claims

## What is Red Teaming?

Red Teaming = **deliberately attacking your own AI** to find weaknesses before real attackers do.

In traditional cybersecurity, companies hire ethical hackers (red teams) to break into their own systems. Same concept for AI.

**How it works:**

```
You write test cases  →  Each test case is an attack
                          ↓
                    Run all attacks on your AI
                          ↓
                    Evaluator checks: did the attack succeed?
                          ↓
                    Report: X/10 attacks succeeded — fix these
```

**We already built this in Section 02** — the Prompt Injection Scanner is a red teaming tool. It fires 8 attacks at your system prompt and scores how many got through.

**Popular red teaming tools:**
- **Promptfoo** — open source, run test suites against any LLM
- **Garak** — specifically for LLM security testing
- **PyRIT** — Microsoft's red teaming toolkit for AI

## What we built in this section

**Project 1: RAG Evaluator (BNS 2024 Law)**

A RAG system over India's new criminal code (Bharatiya Nyaya Sanhita 2024) + a faithfulness evaluator.

```
Question  →  Retrieve BNS sections  →  Generate answer  →  Score faithfulness
```

Why BNS 2024? It replaced the IPC in July 2024 — fresh topic, real consequences if the AI hallucinates legal information.

Files:
- `rag.py` — loads BNS catalog, retrieves relevant sections, generates answer
- `evals.py` — faithfulness() function using LLM judge
- `app.py` — Streamlit UI: pick a question, see answer + faithfulness score

---

**Project 2: Simple AI Answer Grader**

Paste any question + answer → AI judge gives it a score out of 10 with feedback.

Files:
- `backend/main.py` — FastAPI `/grade` endpoint
- `grader_app.py` — Streamlit UI

---

**The security connection:**

| Tool | What it catches |
|------|-----------------|
| Faithfulness evaluator | Hallucinations in RAG output |
| Simple grader | Wrong or misleading answers |
| Prompt injection scanner (Section 02) | Attacks that bypassed your system prompt |